In [1]:
# 1. 베이스 모델 준비
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024    # 한 번에 처리할 최대 토큰 수 (클수록 VRAM 많이 사용)
dtype = None             # GPU에 맞게 자동 선택 (None 권장)
load_in_4bit = True      # 4bit 압축 로드 여부 (True = VRAM 절약)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

print(f"✅ 모델 로드 완료!")
print(f"   dtype : {next(model.parameters()).dtype}")
print(f"   device: {next(model.parameters()).device}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0330 10:38:39.947000 7476 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Sam\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\Sam\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Sam\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4070 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ 모델 로드 완료!
   dtype : torch.bfloat16
   device: cuda:0


In [2]:
# 2. LoRA 어댑터 준비
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # LoRA 크기 (클수록 학습 많이, VRAM 많이 사용)
    target_modules = [           # 어댑터 붙일 레이어 선택
        "q_proj", "k_proj",      #   └─ 어텐션 레이어
        "v_proj", "o_proj",
        "gate_proj", "up_proj",  #   └─ MLP 레이어
        "down_proj",
    ],
    lora_alpha = 16,             # LoRA 학습 강도 (보통 r과 같게 설정)
    lora_dropout = 0,            # 뉴런 드롭 비율 (0 = Unsloth 최적화)
    bias = "none",               # 바이어스 학습 여부 ("none" = Unsloth 최적화)
    use_gradient_checkpointing = "unsloth",  # VRAM 절약 방식 ("unsloth" 권장)
    random_state = 3407,         # 재현성을 위한 랜덤 시드
    use_rslora = False,          # Rank Stabilized LoRA 사용 여부
    loftq_config = None,         # LoftQ 양자화 설정 (실습에서는 불필요)
)

print("✅ LoRA 어댑터 추가 완료!")

Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA 어댑터 추가 완료!


In [3]:
from datasets import load_dataset

dataset = load_dataset("teddylee777/QA-Dataset-mini")
dataset

README.md:   0%|          | 0.00/339 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.20k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 16
    })
})

In [4]:
dataset["train"][0]

{'instruction': '바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?',
 'input': '',
 'output': '바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.'}

In [5]:
dataset["train"][1]

{'instruction': 'G7 국가들이 채택한 AI 국제 행동강령에 따르면, 첨단 AI 시스템의 개발 과정에서 어떤 조치를 취해야 합니까?',
 'input': '',
 'output': 'G7 국가들은 첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 평가 및 완화하는 조치를 채택해야 합니다.'}

In [14]:
dataset = dataset["train"]

In [15]:
def formatting_prompts_func(data):
    instructions = data["instruction"]  # 리스트로 옴
    outputs = data["output"]            # 리스트로 옴

    result = []
    for instruction, output in zip(instructions, outputs):
        messages = [
            {"role": "user",      "content": instruction},
            {"role": "assistant", "content": output},
        ]
        chat_message = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        result.append(chat_message)

    return result  # 리스트 반환

In [11]:
# Trainer를 만들 때 데이터가 이렇게 정돈됩니다.
data = {
    "instruction": [
        '바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?',
        'G7 국가들이 채택한 AI 국제 행동강령에 따르면, 첨단 AI 시스템의 개발 과정에서 어떤 조치를 취해야 합니까?'
    ],
    "output": [
        '바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.',
        'G7 국가들은 첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 평가 및 완화하는 조치를 채택해야 합니다.'
    ]
}

In [8]:
formatting_prompts_func(data)

['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?<|im_end|>\n<|im_start|>assistant\n바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.<|im_end|>\n',
 '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nG7 국가들이 채택한 AI 국제 행동강령에 따르면, 첨단 AI 시스템의 개발 과정에서 어떤 조치를 취해야 합니까?<|im_end|>\n<|im_start|>assistant\nG7 국가들은 첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 평가 및 완화하는 조치를 채택해야 합니다.<|im_end|>\n']

In [16]:
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

args = TrainingArguments(
    per_device_train_batch_size = 2,         # GPU당 배치 크기
    gradient_accumulation_steps = 4,          # 실질 배치 = 2×4 = 8
    warmup_steps = 5,                         # lr 워밍업 스텝 수
    max_steps = 100,                           # 테스트용 (전체 학습 시 num_train_epochs 사용)
    # num_train_epochs = 3,                   # 전체 학습 시 이걸 사용
    learning_rate = 2e-4,                     # LoRA 권장 학습률
    fp16 = not is_bfloat16_supported(),       # 구형 GPU용
    bf16 = is_bfloat16_supported(),           # 최신 GPU용 (자동 감지)
    logging_steps = 1,                        # 몇 스텝마다 로그 출력할지
    optim = "adamw_8bit",                     # VRAM 절약형 옵티마이저
    weight_decay = 0.01,                      # 과적합 방지 정규화
    lr_scheduler_type = "linear",             # lr 감소 방식
    seed = 3407,                              # 재현성을 위한 랜덤 시드
    output_dir = "outputs",                   # 체크포인트 저장 폴더
    report_to = "none",                       # 로그 기록 위치 (wandb 등)
    average_tokens_across_devices = False,    #
)

In [17]:
# KeyError가 나면 위에 dataset = dataset["train"] 코드를 추가해주세요``
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    formatting_func = formatting_prompts_func,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,               # 전처리 CPU 코어 수
    packing = False,                    # 멀티턴 데이터는 False 권장
    args = args,
)

print("✅ 트레이너 설정 완료!")

Unsloth: Tokenizing ["text"]:   0%|          | 0/16 [00:00<?, ? examples/s]

✅ 트레이너 설정 완료!


In [18]:
# 학습 전 GPU 메모리 상태 기록 (학습 후 셀과 비교용)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU 이름     : {gpu_stats.name}")
print(f"전체 VRAM    : {max_memory} GB")
print(f"현재 예약량  : {start_gpu_memory} GB")
print(f"남은 여유    : {round(max_memory - start_gpu_memory, 3)} GB")

GPU 이름     : NVIDIA GeForce RTX 4070 Laptop GPU
전체 VRAM    : 7.996 GB
현재 예약량  : 1.564 GB
남은 여유    : 6.432 GB


In [19]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16 | Num Epochs = 50 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,2.927105
2,2.793565
3,2.788972
4,2.692509
5,2.462417
6,2.042781
7,2.087391
8,1.627831
9,1.533561
10,1.498673


In [20]:
# 학습 후 16bit로 병합 저장
model.save_pretrained_merged(
    "models/model_merged_qwen_teddynote",
    tokenizer,
    save_method="merged_16bit"
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: C:\Users\Sam\.cache\huggingface\hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [08:34<00:00, 514.15s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:05<00:00,  5.29s/it]


Unsloth: Merge process complete. Saved to `C:\Workspaces\local-llm-model-lab\notebooks\models\model_merged_qwen_teddynote`


In [1]:
# 5. 추론
# unsloth Bug : 현재 FastLanguageModel로 추론하는 것은 버그 존재
# 이때 model에 unsloth가 반영되어 있기 때문에 재시작 한 후에 불러와야 한다.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ── 병합된 모델 로드 ──────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    "models/model_merged_qwen_teddynote",
    dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("models/model_merged_qwen_teddynote") # noqa

print("✅ 모델 로드 완료!")

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu126 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0330 11:33:57.158000 25552 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Workspaces\local-llm-model-lab\.venv\Lib\site-packages\torchao\quantization\quant_api.py:1825: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from 'models/model_merged_qwen_teddynote' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ 모델 로드 완료!


In [2]:
input_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "테디노트 유튜브 채널에 대해 알려주세요"}],
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda") # noqa
print(inputs)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198, 130229,  89235, 127121,
          28626, 126310, 144293, 131196,   3315,    109,    226, 139287,  19391,
         131978, 138630,  91669, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [3]:
from transformers import TextStreamer

print("=== 학습 후 출력 (After) ===")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    use_cache=False,
    repetition_penalty=1.3,
    temperature=0.7,
    do_sample=True,
)

=== 학습 후 출력 (After) ===
테디노트(TeddyNote)는 데이터 분석, 머신러닝, 딥러닝 등의 주제를 다루고 있는 유튜브 채널입니다. 이 카운터의 닉네임 'teddonpark'은 테디노츠로 변환되는 플랫폼에서도 사용됩니다. 가수 더치원이 creator ID가기도 합니다. 그와 같은 다양한 강의 내용과 교육 자료を通じ어 AI 및 데이터 분야에서 지식을 공유하고 있습니다<|im_end|>


In [1]:
from unsloth import FastLanguageModel
import torch # noqa

max_seq_length = 1024    # 한 번에 처리할 최대 토큰 수 (클수록 VRAM 많이 사용)
dtype = None             # GPU에 맞게 자동 선택 (None 권장)
load_in_4bit = False      # 4bit 압축 로드 여부 (True = VRAM 절약) ✅

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "models/model_merged_qwen_teddynote",
    max_seq_length = max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

print(f"✅ 모델 로드 완료!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0330 11:50:51.187000 35456 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Sam\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\Sam\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Sam\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4070 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from 'models/model_merged_qwen_teddynote' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'models/model_merged_qwen_teddynote' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ 모델 로드 완료!


In [2]:
# GGUF 변환 + 저장
# cmake installer, visual studio installer, openssl가 자동으로 뜸
# vscode 껏다 켜기
model.save_pretrained_gguf(
    "models/model_merged_gguf",           # 저장 폴더
    tokenizer,
    quantization_method = "q4_k_m"        # 양자화 방식
)

Unsloth: Model is not a PEFT model. Using existing checkpoint at models/model_merged_qwen_teddynote
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: llama.cpp folder exists but binaries not found - will build
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['models/model_merged_qwen_teddynote_gguf\\model_merged_qwen_teddynote.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['models/model_merged_qwen_teddynote_gguf\\model_merged_qwen_teddynote.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'models/model_merged_qwen_teddynote'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: C:\Users\Sam\.unsloth\llama.cpp\build\bin\Release\llama-cli.exe --model models/model_merged_qwen_teddynote_gguf\model_merged_qwen_teddynote.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': 'models/model_merged_qwen_teddynote',
 'gguf_directory': 'models/model_merged_qwen_teddynote_gguf',
 'gguf_files': ['models/model_merged_qwen_teddynote_gguf\\model_merged_qwen_teddynote.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}